# PostgreSQL Database Checker

This notebook helps you verify your PostgreSQL database setup before running Airflow DAGs.

## Setup Instructions
1. Update the database configuration in the next cell with your actual credentials
2. Run all cells to check your database setup
3. Fix any issues before running your Airflow DAGs


In [ ]:
# Database connection parameters
# TODO: Update these with your actual database details
DB_CONFIG = {
    'host': 'YOUR_PUBLIC_IP_HERE',  # Replace with your PostgreSQL public IP
    'port': 5432,
    'database': 'postgres',  # or your database name
    'user': 'postgres',  # or your username
    'password': 'YOUR_PASSWORD_HERE'  # Replace with your password
}

print("Database configuration:")
print(f"Host: {DB_CONFIG['host']}")
print(f"Port: {DB_CONFIG['port']}")
print(f"Database: {DB_CONFIG['database']}")
print(f"User: {DB_CONFIG['user']}")
print(f"Password: {'*' * len(DB_CONFIG['password'])}")


In [ ]:
# Import required libraries
import psycopg2
import pandas as pd
from sqlalchemy import create_engine
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")


In [ ]:
def get_connection():
    """Create a connection to PostgreSQL database"""
    try:
        conn = psycopg2.connect(**DB_CONFIG)
        return conn
    except Exception as e:
        print(f"Error connecting to database: {e}")
        return None

def get_sqlalchemy_engine():
    """Create SQLAlchemy engine for pandas operations"""
    try:
        connection_string = f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
        engine = create_engine(connection_string)
        return engine
    except Exception as e:
        print(f"Error creating SQLAlchemy engine: {e}")
        return None

print("Connection functions defined!")


## 1. Test Database Connection


In [ ]:
# Test database connection
print("Testing database connection...")
conn = get_connection()

if conn:
    print("✅ Database connection successful!")
    
    # Get database version
    cursor = conn.cursor()
    cursor.execute("SELECT version();")
    version = cursor.fetchone()[0]
    print(f"Database version: {version.split()[0]} {version.split()[1]}")
    
    conn.close()
else:
    print("❌ Database connection failed!")
    print("Please check your DB_CONFIG settings.")


## 2. Check Schemas


In [ ]:
# Check if required schemas exist
print("Checking schemas...")

conn = get_connection()
if conn:
    cursor = conn.cursor()
    cursor.execute("""
        SELECT schema_name 
        FROM information_schema.schemata 
        WHERE schema_name IN ('bronze', 'silver', 'gold', 'snapshots')
        ORDER BY schema_name;
    """)
    
    schemas = cursor.fetchall()
    required_schemas = ['bronze', 'silver', 'gold', 'snapshots']
    
    print(f"Required schemas: {required_schemas}")
    print(f"Existing schemas: {[schema[0] for schema in schemas]}")
    
    missing_schemas = set(required_schemas) - set([schema[0] for schema in schemas])
    
    if missing_schemas:
        print(f"❌ Missing schemas: {missing_schemas}")
        print("\nTo create missing schemas, run:")
        for schema in missing_schemas:
            print(f"CREATE SCHEMA IF NOT EXISTS {schema};")
    else:
        print("✅ All required schemas exist!")
    
    conn.close()
else:
    print("❌ Could not connect to database")


## 3. Check Bronze Tables


In [ ]:
# Check if bronze tables exist
print("Checking bronze tables...")

conn = get_connection()
if conn:
    cursor = conn.cursor()
    
    # Check if bronze tables exist
    bronze_tables = ['raw_census_g01', 'raw_census_g02', 'raw_lga_mapping', 'raw_airbnb_listings']
    
    cursor.execute("""
        SELECT table_name 
        FROM information_schema.tables 
        WHERE table_schema = 'bronze' 
        AND table_name IN %s
        ORDER BY table_name;
    """, (tuple(bronze_tables),))
    
    existing_tables = [row[0] for row in cursor.fetchall()]
    print(f"Required bronze tables: {bronze_tables}")
    print(f"Existing bronze tables: {existing_tables}")
    
    missing_tables = set(bronze_tables) - set(existing_tables)
    if missing_tables:
        print(f"❌ Missing bronze tables: {missing_tables}")
        print("\nTo create missing tables, run the create_bronze_tables.sql file")
    else:
        print("✅ All bronze tables exist!")
    
    conn.close()
else:
    print("❌ Could not connect to database")


## 4. Check Table Data


In [ ]:
# Check if tables have data
print("Checking table data...")

engine = get_sqlalchemy_engine()
if engine:
    bronze_tables = ['raw_census_g01', 'raw_census_g02', 'raw_lga_mapping', 'raw_airbnb_listings']
    
    for table in bronze_tables:
        try:
            df = pd.read_sql(f"SELECT COUNT(*) as count FROM bronze.{table}", engine)
            count = df['count'].iloc[0]
            print(f"{table}: {count} rows")
            
            if count > 0:
                # Show sample data
                sample_df = pd.read_sql(f"SELECT * FROM bronze.{table} LIMIT 2", engine)
                print(f"  Sample columns: {list(sample_df.columns)[:5]}...")
                print(f"  Sample data shape: {sample_df.shape}")
                
        except Exception as e:
            print(f"❌ Error checking {table}: {e}")
else:
    print("❌ Could not create SQLAlchemy engine")


## 5. Summary Check


In [ ]:
# Summary check
print("=" * 60)
print("DATABASE STATUS SUMMARY")
print("=" * 60)

conn = get_connection()
if conn:
    cursor = conn.cursor()
    
    # Check connection
    print("✅ Database connection: OK")
    
    # Check schemas
    cursor.execute("""
        SELECT COUNT(*) FROM information_schema.schemata 
        WHERE schema_name IN ('bronze', 'silver', 'gold', 'snapshots');
    """)
    schema_count = cursor.fetchone()[0]
    print(f"✅ Required schemas: {schema_count}/4")
    
    # Check bronze tables
    cursor.execute("""
        SELECT COUNT(*) FROM information_schema.tables 
        WHERE table_schema = 'bronze' 
        AND table_name IN ('raw_census_g01', 'raw_census_g02', 'raw_lga_mapping', 'raw_airbnb_listings');
    """)
    table_count = cursor.fetchone()[0]
    print(f"✅ Bronze tables: {table_count}/4")
    
    # Check data
    bronze_tables = ['raw_census_g01', 'raw_census_g02', 'raw_lga_mapping', 'raw_airbnb_listings']
    total_rows = 0
    for table in bronze_tables:
        try:
            cursor.execute(f"SELECT COUNT(*) FROM bronze.{table}")
            count = cursor.fetchone()[0]
            total_rows += count
        except:
            pass
    
    print(f"✅ Total rows in bronze tables: {total_rows}")
    
    if schema_count == 4 and table_count == 4:
        print("\n🎉 DATABASE IS READY FOR AIRFLOW!")
    else:
        print("\n⚠️  DATABASE NEEDS SETUP BEFORE RUNNING AIRFLOW")
        print("Run the 'Create Bronze Tables' section above.")
    
    conn.close()
else:
    print("❌ Database connection failed!")

print("=" * 60)
